In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# DATA UNDERSTANDING

In [ ]:
df = pd.read_csv("phone_addiction.csv")
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

In [ ]:
df = df.drop(columns=['ID', 'Name'])

# DATA CLEANING

In [ ]:
df["Addiction_Level_Class"] = pd.cut(df["Addiction_Level"], bins=3,labels=["Low", "Medium", "High"])

In [ ]:
print("Number of duplicate rows:", df.duplicated().sum())

In [ ]:
print("\nUnique Values per Column:\n", df.nunique())

In [ ]:
# Check for missing values
print(f" Dataset Shape: {df.shape}")
df.isnull().sum()

#### Handling Missing Values

In [ ]:
# Fill numeric missing values using median
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Fill categorical missing values using mode
cat_cols = df.select_dtypes(include=['object', 'category']).columns
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])

#### Remove Outliers using IQR method

In [ ]:
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = np.where(df[col] < lower, lower,
               np.where(df[col] > upper, upper, df[col]))

In [ ]:
df[num_cols].hist(figsize=(30,20), bins=20, color='skyblue', edgecolor='black')
plt.suptitle('Numerical Feature Distributions')
plt.show()

In [ ]:
# Bar plot for a categorical feature (example: Gender)
df['Gender'].value_counts().plot(kind='bar')
plt.title("Gender Distribution")
plt.show()

# 1. UNIVARIATE ANALYSIS

#### Distribution of Daily Phone Usage
- Insight: Detects high-usage students who may be at addiction risk.

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df["Daily_Usage_Hours"], kde=True, bins=30)
plt.title("Distribution of Daily Phone Usage Hours")
plt.xlabel("Daily Phone Usage (hours)")
plt.ylabel("Count")
plt.show()

#### Anxiety Level Distribution
- Insight: Helps identify if anxiety correlates with addiction later.

In [ ]:
df["Anxiety_Level"].value_counts().sort_index().plot(kind="bar", figsize=(7,4))
plt.title("Distribution of Anxiety Levels")
plt.xlabel("Anxiety Level")
plt.ylabel("Number of Students")
plt.show()

#### Sleep Hours Data
- Insight: Identifies low-sleep students which may be linked to addiction.

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x=df["Sleep_Hours"])
plt.title("Boxplot of Sleep Hours")
plt.xlabel("Sleep Hours")
plt.show()

# 2. Bi-Variate Analysis

#### Relationship Between Phone Usage & Academic Performance
- Insight: See if more phone usage means lower grades.

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(x=df["Daily_Usage_Hours"],y=df["Academic_Performance"])
plt.title("Daily Usage vs Academic Performance")
plt.xlabel("Daily Usage Hours")
plt.ylabel("Academic Performance")
plt.show()

#### Gender vs Phone Addiction Level (Bar Plot)
- Insight: Compare addiction tendency by gender.

In [ ]:
plt.figure(figsize=(12,6))
sns.barplot(x="Gender", y="Addiction_Level", data=df)
plt.title("Average Addiction Level by Gender")
plt.xlabel("Gender")
plt.ylabel("Addiction Level")
plt.show()

#### Sleep Hours vs Anxiety Level (Boxplot)
- Insight: Shows how anxiety might affect sleep duration.

In [ ]:
plt.figure(figsize=(8,4))
sns.boxplot(x="Anxiety_Level", y="Sleep_Hours", data=df)
plt.title("Sleep Hours by Anxiety Level")
plt.xlabel("Anxiety Level")
plt.ylabel("Sleep Hours")
plt.show()

# 3. Multi-Variate

#### Usage + Sleep vs Academic Performance
- Insight: Combined effect of usage & sleep on academics.

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(x="Daily_Usage_Hours", y="Academic_Performance",hue="Sleep_Hours",size="Addiction_Level",data=df, palette="viridis")
plt.title("Usage, Sleep & Addiction vs Academic Performance")
plt.xlabel("Daily Usage Hours")
plt.ylabel("Academic Performance")
plt.show()

#### Anxiety vs Sleep vs Addiction Level
- Helps you identify which pair of features are strongly related.

In [ ]:
features = ["Daily_Usage_Hours", "Sleep_Hours", "Anxiety_Level", "Academic_Performance"]

plt.figure(figsize=(7,4))
sns.heatmap(df[features].corr(), annot=True, cmap="Blues")
plt.title("Correlation Between Usage, Sleep, Anxiety & Academics")
plt.show()

# Data Preprocessing

#### split data into train and test 

In [ ]:
target = "Addiction_Level_Class"
le = LabelEncoder()
y = le.fit_transform(df[target])
X = df.drop(columns=[target])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)

In [ ]:
# Scale numerical columns
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Model Training

#### 1. Logistic Regression

In [ ]:
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_test_scaled)

print("\n=== Logistic Regression ===")
print(classification_report(y_test, y_pred_lr))

#### 2. Support Vector Machine (SVM)

In [ ]:
svm_model = SVC()
svm_model.fit(X_train_scaled, y_train)
y_pred_svm = svm_model.predict(X_test_scaled)

print("\n=== SVM Classifier ===")
print(classification_report(y_test, y_pred_svm))

#### 3. KNN Classifier

In [ ]:
knn = KNeighborsClassifier()
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)

print("\n=== KNN Classifier ===")
print(classification_report(y_test, y_pred_knn))

#### 4. Decision Tree Classifier

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train_scaled, y_train)
y_pred_dt = dt.predict(X_test_scaled)

print("\n=== Decision Tree ===")
print(classification_report(y_test, y_pred_dt))

#### 5. Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

print("\n=== Random Forest ===")
print(classification_report(y_test, y_pred_rf))


#### 6. XGBoost Classifier

In [ ]:
xgb = XGBClassifier(eval_metric="mlogloss", random_state=42, stratify = 'y')
xgb.fit(X_train_scaled, y_train)
y_pred_xgb = xgb.predict(X_test_scaled)

print("\n=== XGBoost Classifier ===")
print(classification_report(y_test, y_pred_xgb))